#Import Libraries

In [14]:
# ================================================
# SAUDI RETAIL SALES FORECASTING
# Notebook 2: SQL Analysis
# ================================================

import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries ready!")

✅ Libraries ready!


#Load Clean CSV

In [15]:
# Load the clean CSV from Notebook 1
df_clean = pd.read_csv("/content/saudi_sales_clean.csv",
                        parse_dates=['date'])

print("✅ Clean data loaded!")
print(f"📊 Rows: {df_clean.shape[0]}")
print(f"📅 Date: {df_clean['date'].min()} → {df_clean['date'].max()}")
df_clean.head()

✅ Clean data loaded!
📊 Rows: 135
📅 Date: 2015-01-01 00:00:00 → 2026-03-01 00:00:00


,date,sales_thousand_sar,year,month,month_name,quarter
0,2015-01-01,1.422696e+07,2015,1,January,1
1,2015-02-01,1.535275e+07,2015,2,February,1
2,2015-03-01,1.540108e+07,2015,3,March,1
3,2015-04-01,1.356617e+07,2015,4,April,2
4,2015-05-01,1.514562e+07,2015,5,May,2


#Create SQL Database

In [16]:
# Create SQL database in memory
conn = sqlite3.connect(":memory:")
df_clean.to_sql("pos_sales", conn,
                if_exists="replace", index=False)

verify = pd.read_sql("SELECT COUNT(*) as rows FROM pos_sales", conn)
print("✅ SQL Database ready!")
print(f"📋 Rows in database: {verify['rows'][0]}")

✅ SQL Database ready!
📋 Rows in database: 135


Q1: Yearly Sales

In [17]:
q1 = pd.read_sql("""
    SELECT
        year,
        ROUND(SUM(sales_thousand_sar)/1000000, 2) AS total_sales_billion_sar,
        COUNT(*) AS months_recorded
    FROM pos_sales
    GROUP BY year
    ORDER BY year
""", conn)

print("=" * 50)
print("📊 Q1: Total Sales Per Year (Billion SAR)")
print("=" * 50)
print(q1.to_string(index=False))
print(f"\n💡 Sales grew from {q1['total_sales_billion_sar'].iloc[0]}B")
print(f"   to {q1['total_sales_billion_sar'].iloc[-1]}B SAR!")

📊 Q1: Total Sales Per Year (Billion SAR)
 year  total_sales_billion_sar  months_recorded
 2015                   172.84               12
 2016                   182.75               12
 2017                   200.47               12
 2018                   232.31               12
 2019                   287.79               12
 2020                   357.30               12
 2021                   473.26               12
 2022                   559.13               12
 2023                   613.96               12
 2024                   668.18               12
 2025                   707.15               12
 2026                   189.71                3

💡 Sales grew from 172.84B
   to 189.71B SAR!


Q2: Best Months

In [18]:
q2 = pd.read_sql("""
    SELECT
        month,
        month_name,
        ROUND(AVG(sales_thousand_sar)/1000000, 2) AS avg_sales_billion,
        ROUND(MAX(sales_thousand_sar)/1000000, 2) AS max_sales_billion,
        ROUND(MIN(sales_thousand_sar)/1000000, 2) AS min_sales_billion
    FROM pos_sales
    GROUP BY month, month_name
    ORDER BY avg_sales_billion DESC
""", conn)

print("=" * 55)
print("📊 Q2: Best Months by Average Sales")
print("=" * 55)
print(q2.to_string(index=False))
print(f"\n💡 Busiest month: {q2['month_name'].iloc[0]}")
print(f"   Slowest month: {q2['month_name'].iloc[-1]}")

📊 Q2: Best Months by Average Sales
 month month_name  avg_sales_billion  max_sales_billion  min_sales_billion
     3      March              37.80              66.14              15.36
    12   December              37.10              61.14              14.89
     8     August              35.22              62.57              14.67
    11   November              34.73              59.01              13.96
     1    January              34.66              63.69              14.23
    10    October              34.20              59.86              12.54
     5        May              34.04              59.50              15.15
     6       June              33.88              53.98              16.06
     9  September              33.66              58.24              12.55
     7       July              33.14              58.89              13.57
     2   February              32.48              59.88              13.48
     4      April              31.81              52.95          

Q3: Year over Year Growth

In [19]:
q3 = pd.read_sql("""
    WITH yearly AS (
        SELECT year,
               ROUND(SUM(sales_thousand_sar)/1000000, 2) AS total_sales,
               COUNT(*) AS months_count
        FROM pos_sales
        GROUP BY year
    )
    SELECT
        year,
        total_sales          AS sales_billion_sar,
        months_count,
        CASE
            WHEN months_count < 12
            THEN 'Incomplete Year'
            ELSE 'Complete Year'
        END                  AS year_status,
        ROUND(
            ((total_sales - LAG(total_sales) OVER (ORDER BY year))
            / LAG(total_sales) OVER (ORDER BY year)) * 100, 2
        )                    AS growth_percent
    FROM yearly
    ORDER BY year
""", conn)

print("=" * 65)
print("📊 Q3: Year over Year Growth % (Complete Years Only)")
print("=" * 65)
print(q3.to_string(index=False))

# Only analyze complete years
complete_years = q3[q3['year_status'] == 'Complete Year'].dropna()
incomplete     = q3[q3['year_status'] == 'Incomplete Year']

best  = complete_years.loc[complete_years['growth_percent'].idxmax()]
worst = complete_years.loc[complete_years['growth_percent'].idxmin()]

print(f"\n💡 Best growth year:  {int(best['year'])} → +{best['growth_percent']}%")
print(f"💡 Worst growth year: {int(worst['year'])} → {worst['growth_percent']}%")
print(f"\n⚠️  Incomplete years (excluded from growth analysis):")
for _, row in incomplete.iterrows():
    print(f"   → {int(row['year'])}: only {int(row['months_count'])} months recorded")

📊 Q3: Year over Year Growth % (Complete Years Only)
 year  sales_billion_sar  months_count     year_status  growth_percent
 2015             172.84            12   Complete Year             NaN
 2016             182.75            12   Complete Year            5.73
 2017             200.47            12   Complete Year            9.70
 2018             232.31            12   Complete Year           15.88
 2019             287.79            12   Complete Year           23.88
 2020             357.30            12   Complete Year           24.15
 2021             473.26            12   Complete Year           32.45
 2022             559.13            12   Complete Year           18.14
 2023             613.96            12   Complete Year            9.81
 2024             668.18            12   Complete Year            8.83
 2025             707.15            12   Complete Year            5.83
 2026             189.71             3 Incomplete Year          -73.17

💡 Best growth year:  202

Q4: Top 10 Months Ever

In [20]:
q4 = pd.read_sql("""
    SELECT
        date, year, month_name, quarter,
        ROUND(sales_thousand_sar/1000000, 2) AS sales_billion_sar
    FROM pos_sales
    ORDER BY sales_thousand_sar DESC
    LIMIT 10
""", conn)

print("=" * 55)
print("📊 Q4: Top 10 Highest Sales Months Ever")
print("=" * 55)
print(q4.to_string(index=False))
print(f"\n💡 Best month ever: {q4['month_name'].iloc[0]} {int(q4['year'].iloc[0])}")
print(f"   Sales: {q4['sales_billion_sar'].iloc[0]} Billion SAR!")

📊 Q4: Top 10 Highest Sales Months Ever
               date  year month_name  quarter  sales_billion_sar
2026-03-01 00:00:00  2026      March        1              66.14
2025-03-01 00:00:00  2025      March        1              65.66
2026-01-01 00:00:00  2026    January        1              63.69
2025-08-01 00:00:00  2025     August        3              62.57
2025-12-01 00:00:00  2025   December        4              61.14
2026-02-01 00:00:00  2026   February        1              59.88
2025-10-01 00:00:00  2025    October        4              59.86
2024-03-01 00:00:00  2024      March        1              59.68
2025-05-01 00:00:00  2025        May        2              59.50
2024-12-01 00:00:00  2024   December        4              59.09

💡 Best month ever: March 2026
   Sales: 66.14 Billion SAR!


 Q5: Quarterly Performance

In [21]:
q5 = pd.read_sql("""
    SELECT
        quarter,
        CASE quarter
            WHEN 1 THEN 'Q1: Jan-Feb-Mar'
            WHEN 2 THEN 'Q2: Apr-May-Jun'
            WHEN 3 THEN 'Q3: Jul-Aug-Sep'
            WHEN 4 THEN 'Q4: Oct-Nov-Dec'
        END AS quarter_name,
        ROUND(AVG(sales_thousand_sar)/1000000, 2) AS avg_sales_billion,
        ROUND(SUM(sales_thousand_sar)/1000000, 2) AS total_sales_billion
    FROM pos_sales
    GROUP BY quarter
    ORDER BY avg_sales_billion DESC
""", conn)

print("=" * 50)
print("📊 Q5: Performance by Quarter")
print("=" * 50)
print(q5.to_string(index=False))
print(f"\n💡 Best quarter:  {q5['quarter_name'].iloc[0]}")
print(f"💡 Worst quarter: {q5['quarter_name'].iloc[-1]}")

📊 Q5: Performance by Quarter
 quarter    quarter_name  avg_sales_billion  total_sales_billion
       4 Q4: Oct-Nov-Dec              35.34              1166.35
       1 Q1: Jan-Feb-Mar              34.98              1259.22
       3 Q3: Jul-Aug-Sep              34.01              1122.20
       2 Q2: Apr-May-Jun              33.24              1097.08

💡 Best quarter:  Q4: Oct-Nov-Dec
💡 Worst quarter: Q2: Apr-May-Jun


 Save All Results

In [22]:
# Save all SQL results for Power BI later
q1.to_csv("/content/sql_yearly_sales.csv",    index=False)
q2.to_csv("/content/sql_monthly_sales.csv",   index=False)
q3.to_csv("/content/sql_yoy_growth.csv",      index=False)
q4.to_csv("/content/sql_top10_months.csv",    index=False)
q5.to_csv("/content/sql_quarterly_sales.csv", index=False)

print("✅ All SQL results saved!")
print("\n📁 Files saved:")
print("   → sql_yearly_sales.csv")
print("   → sql_monthly_sales.csv")
print("   → sql_yoy_growth.csv")
print("   → sql_top10_months.csv")
print("   → sql_quarterly_sales.csv")
print("\n🎯 Notebook 2 COMPLETE!")
print("👉 Next: Notebook 3 - Visualization!")

✅ All SQL results saved!

📁 Files saved:
   → sql_yearly_sales.csv
   → sql_monthly_sales.csv
   → sql_yoy_growth.csv
   → sql_top10_months.csv
   → sql_quarterly_sales.csv

🎯 Notebook 2 COMPLETE!
👉 Next: Notebook 3 - Visualization!
